# Walkthrough · the epigraphist: Greek inscriptions end to end

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryanpavlicek/pyaegean/blob/main/notebooks/walkthrough-epigraphist.ipynb)

*You work on Greek inscriptions: stones with gaps, editorial restorations, and a find-site you care about.* This notebook is one working session with pyaegean's epigraphic side, following [Recipes → workflow A](https://github.com/ryanpavlicek/pyaegean/wiki/Recipes) (the epigraphist).

By the end you will have:

* an inscription corpus loaded, with the editor's per-token **reading status** (on the stone, unclear, restored, lost) preserved;
* an EpiDoc TEI edition of your own imported next to it;
* a filtered, **citable subset** exported to token-level CSV and to EpiDoc, then read back to verify;
* a reproducibility trail: package version, content fingerprint, citation.

**Offline-first.** Every ungated cell runs with no network. The real epigraphy corpora fetch on demand, so the cells that need one are gated behind the `RUN_HEAVY` switch below; when it is off, the small bundled Greek sample corpus stands in (same API, same commands, a clean literary text instead of stones).

In [ ]:
# pyaegean works fully offline once installed. This installs it the first time
# you open the notebook in Colab; locally it's a no-op if it's already present.
try:
    import aegean
except ImportError:
    %pip install -q pyaegean
    import aegean

print("pyaegean", aegean.__version__)

In [ ]:
# ── The one switch for the optional, downloading cells ──────────────────────
# Leave this False to keep the whole notebook offline: the bundled Greek sample
# corpus stands in for the fetched inscriptions. Flip it to True (on decent
# wifi) to work on the real corpus:
#   * aegean.load("isicily")   ~7 MB one-time fetch — I.Sicily, 2,855 Greek
#                              inscriptions of Sicily (CC BY 4.0)
# It fetches once, then caches. Every other cell runs offline with no downloads.
RUN_HEAVY = False

print("Heavy/optional cells are",
      "ON — first run downloads." if RUN_HEAVY
      else "OFF — the offline core still runs in full.")

## Step 1 · Load an inscription corpus

`aegean.load("isicily")` fetches I.Sicily once (a sha256-pinned asset, CC BY 4.0), then loads from the local store. Four more Greek epigraphy corpora load exactly the same way: `iip` (Israel/Palestine), `iospe` (northern Black Sea coast), `igcyr` (Cyrenaica), and `edh` (the Greek subset of the Epigraphic Database Heidelberg). For documentary papyri at scale, the `ddbdp` database is its own workflow ([Recipes → workflow B](https://github.com/ryanpavlicek/pyaegean/wiki/Recipes)).

With the switch off, the bundled `greek` sample keeps every later cell runnable: five short literary passages, zero network.

In [ ]:
if RUN_HEAVY:
    corpus = aegean.load("isicily")   # 2,855 Greek inscriptions of Sicily, fetched once
else:
    corpus = aegean.load("greek")     # bundled sample passages: same API, zero network
    print("[offline] using the bundled 'greek' sample — set RUN_HEAVY = True for I.Sicily")

print(len(corpus.documents), "documents ·", sum(len(d.tokens) for d in corpus.documents), "tokens")
for d in corpus.documents[:3]:
    print(f"  {d.id:22} · {d.meta.name or d.meta.site} · {d.meta.period}")
# offline —
#   5 documents · 27 tokens
#   iliad-1.1              · Iliad 1.1 · Archaic (epic)
#   john-1.1               · John 1:1 · Koine
#   heraclitus-panta-rhei  · Fragment (panta rhei) · Archaic
# with RUN_HEAVY —
#   2855 documents · 28919 tokens
#   ISic000046             · I.Sicily inscription 000046 · 5th century CE(?)

## Step 2 · The editorial apparatus: `ReadingStatus`

Epigraphic text is not all equally *there*, and pretending otherwise corrupts every count downstream. The epigraphy corpora keep the editor's apparatus as a per-token `ReadingStatus`:

| status | meaning |
|--------|---------|
| `CERTAIN` | read on the stone |
| `UNCLEAR` | damaged but read (underdotted in Leiden notation) |
| `RESTORED` | supplied by the editor (square brackets) |
| `LOST` | in a lacuna, not recovered |

The corpus-level `provenance.edition_fidelity` flag records how the text relates to the printed edition (for I.Sicily: `apparatus-preserved,normalized`). The bundled literary sample has no apparatus, so there everything is `CERTAIN`; the cell below reports whichever corpus you loaded.

In [ ]:
from collections import Counter

dist = Counter(t.status.name for d in corpus.documents for t in d.tokens)
print("reading statuses:", dict(dist.most_common()))
print("edition fidelity:", corpus.provenance.edition_fidelity or "(not an apparatus corpus)")
# offline —
#   reading statuses: {'CERTAIN': 27}
#   edition fidelity: (not an apparatus corpus)
# with RUN_HEAVY —
#   reading statuses: {'CERTAIN': 22124, 'RESTORED': 5340, 'UNCLEAR': 1143, 'LOST': 312}
#   edition fidelity: apparatus-preserved,normalized

In [ ]:
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to read an I.Sicily inscription with its apparatus.")
else:
    d = corpus.get("ISic001480")      # a Kamarina inscription with damage and repair
    print(d.id, "·", d.meta.site, "·", d.meta.period)
    for t in d.tokens:
        print(f"  {t.text:14} {t.status.name}")
    # ISic001480 · Kamarina · 4th or 2nd century BCE
    #   ΔΥΛΙΙΑΩΝ       UNCLEAR    ← damaged letters, read with hesitation
    #   υἱὸς           CERTAIN
    #   Κόρυδός        CERTAIN
    #   με             CERTAIN
    #   Γέλων          RESTORED   ← supplied by the editor
    #   ἐποίησε        CERTAIN

## Step 3 · Bring your own EpiDoc (offline)

The same apparatus arrives from editions pyaegean does not ship. Any EpiDoc TEI file whose tokens are carried by `<w>`/`<num>`/`<g>` elements imports directly: `<supplied>` becomes `RESTORED`, `<unclear>` becomes `UNCLEAR`. This cell writes a miniature edition to disk and imports it, entirely offline.

In [ ]:
import os
import tempfile

from aegean import io

EPIDOC = """<?xml version="1.0" encoding="UTF-8"?>
<TEI xmlns="http://www.tei-c.org/ns/1.0">
 <teiHeader><fileDesc><titleStmt><title>Funerary epigram</title></titleStmt>
  <publicationStmt><idno>MyColl 1</idno></publicationStmt>
  <sourceDesc><msDesc><history><origin><origPlace>Kamarina</origPlace></origin></history></msDesc></sourceDesc>
 </fileDesc></teiHeader>
 <text><body>
  <div type="edition" xml:lang="grc">
   <ab>
    <lb n="1"/><w>Ἀρτεμιδώρα</w> <w><unclear>χρηστὰ</unclear></w>
    <lb n="2"/><w><supplied reason="lost">χαῖρε</supplied></w>
   </ab>
  </div>
 </body></text>
</TEI>
"""

workdir = tempfile.mkdtemp()          # keep the demo self-contained; use your own paths
xml_path = os.path.join(workdir, "myinscription.xml")
with open(xml_path, "w", encoding="utf-8") as f:
    f.write(EPIDOC)

mine = io.from_epidoc(xml_path, script_id="greek")
d = mine.documents[0]
print(d.id, "· site:", d.meta.site)
for t in d.tokens:
    print(f"  {t.text:14} {t.status.name}")
print(mine.cite())
# MyColl 1 · site: Kamarina
#   Ἀρτεμιδώρα     CERTAIN
#   χρηστὰ         UNCLEAR
#   χαῖρε          RESTORED
# EpiDoc TEI import: myinscription.xml   ← the provenance honestly records a local import

## Step 4 · Filter down to the documents that matter

The query engine takes typed filter rows (`aegean query CORPUS --fields` lists each corpus's vocabulary) and returns a result that remembers what was asked; `to_corpus` turns the hits into a real corpus whose citation records the query. On I.Sicily you would as often start from a find-site filter (`corpus.filter(site="Kamarina")`): the gated cell after this one does exactly that.

In [ ]:
from aegean.analysis import FilterRow

word = "τετάρτα" if RUN_HEAVY else "λόγος"   # a Doric phratry ordinal, or a sample word
res = corpus.query([FilterRow("ins-contains-word", word)])
print(res.description, "→", len(res.inscriptions), "document(s)")
for d in res.inscriptions[:3]:
    print("  ", d.id, "·", d.meta.site or d.meta.name)

subset = res.to_corpus(corpus)               # a real Corpus that remembers the query
print(subset.cite())
# offline —
#   Contains exact word: λόγος → 1 document(s)
#      john-1.1 · John 1:1
#   Homer, Herodotus, Heraclitus, Sappho, Gospel of John (sample excerpts).
#     [subset: query(Contains exact word: λόγος) → 1 documents]
# with RUN_HEAVY —
#   Contains exact word: τετάρτα → 15 document(s)
#      ISic030030 · Kamarina  (and 14 more phratry tablets)

In [ ]:
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True for the find-site filter and keyness on I.Sicily.")
else:
    from aegean.analysis import keyness

    kamarina = corpus.filter(site="Kamarina")
    rest = [d for d in corpus.documents if d.meta.site != "Kamarina"]
    print(len(kamarina.documents), "documents from Kamarina")
    for r in keyness(kamarina, rest)[:5]:
        print(f"  {r.item:10} {r.target_count:>3}×  G²={r.log_likelihood:7.2f}  log-ratio={r.log_ratio:+.2f}")
    # 174 documents from Kamarina
    #   τετάρτα     16×  G²= 104.63  log-ratio=+9.70
    #   φράτρα      14×  G²=  91.53  log-ratio=+9.51
    #   τα          17×  G²=  76.36  log-ratio=+5.42
    #   ευτέρα      10×  G²=  65.34  log-ratio=+9.04
    #   δεκάτα       9×  G²=  58.80  log-ratio=+8.90
    # φράτρα and the Doric feminine ordinals: the city's phratry tablets speaking.
    # Text is kept as inscribed, so fragmentary tokens (τα, ευτέρα) rank alongside
    # whole words. Inspect before you interpret.

## Step 5 · Export, and read back what you exported

The token-level CSV carries the reading-status column, so a spreadsheet analysis can weigh restored text differently from text on the stone. The EpiDoc export writes one TEI file per document; reading the files back into a corpus verifies the trip was lossless where it matters.

In [ ]:
csv_path = os.path.join(workdir, "subset_tokens.csv")
io.to_csv(subset, csv_path, level="token")        # one row per token, status included
with open(csv_path, encoding="utf-8") as f:
    for line in f.read().splitlines()[:4]:
        print(" ", line)
# offline —
#   doc_id,line_no,position,text,kind,status,site,period
#   john-1.1,,0,ἐν,word,certain,,Koine
#   john-1.1,,1,ἀρχῇ,word,certain,,Koine
#   john-1.1,,2,ἦν,word,certain,,Koine

epi_dir = os.path.join(workdir, "epidoc")
io.write_epidoc(subset, epi_dir)                  # one TEI file per document
back = io.from_epidoc(epi_dir, script_id="greek")
print("EpiDoc round trip:", len(back.documents), "document(s) re-read;",
      "first tokens:", " ".join(t.text for t in back.documents[0].tokens[:5]))
# EpiDoc round trip: 1 document(s) re-read; first tokens: ἐν ἀρχῇ ἦν ὁ λόγος

## Step 6 · The reproducibility trail

Three lines pin exactly what your numbers were computed on: the package version, a stable content fingerprint of the data, and a citation that names the source, the license, and the query that carved the subset. Paste them into a methods section and the analysis is reconstructible.

In [ ]:
print("pyaegean", aegean.__version__)
print("fingerprint:", subset.fingerprint()[:16])   # content hash: same data, same hash
print(subset.cite())
# offline —
#   fingerprint: 2969ab9b5f904289
#   Homer, Herodotus, Heraclitus, Sappho, Gospel of John (sample excerpts).
#     [subset: query(Contains exact word: λόγος) → 1 documents]
# with RUN_HEAVY —
#   fingerprint: 85fda29ae0653460
#   I.Sicily (ISicily/ISicily, CC BY 4.0), primary-Greek inscriptions
#     — https://github.com/ISicily/ISicily
#     [subset: query(Contains exact word: τετάρτα) → 15 documents]

## Where to go next

* [Recipes](https://github.com/ryanpavlicek/pyaegean/wiki/Recipes): workflow A chains these same moves from the command line (`aegean load isicily --site Kamarina`, `aegean keyness isicily --site Kamarina --top 5`, `aegean export … -f csv --level token`); workflow B covers the DDbDP papyri.
* [Using Critical Editions](https://github.com/ryanpavlicek/pyaegean/wiki/Using-Critical-Editions): the reading-status and edition-fidelity model in depth.
* [Data & Provenance](https://github.com/ryanpavlicek/pyaegean/wiki/Data-and-Provenance): every corpus's license, source, and sha256 pin.
* [Choosing a Workflow](https://github.com/ryanpavlicek/pyaegean/wiki/Choosing-a-Workflow) and [Choosing a Pipeline](https://github.com/ryanpavlicek/pyaegean/wiki/Choosing-a-Pipeline): match the corpus and backend to your goal.
* [CLI](https://github.com/ryanpavlicek/pyaegean/wiki/CLI): the full command reference.